# Module 3 Group Discussion: Working with Pandas DataFrames

This notebook uses a small restaurant-operations dataset to walk through the discussion process in **GroupDiscussion_IntroPythonProgramming_v1.docx**.

## Learning goals
- Load a CSV file
- Inspect and understand a DataFrame
- Clean missing values and inconsistent text
- Convert and analyze dates
- Select and filter rows and columns
- Create and update columns safely
- Build a simple management summary

> Run the cells from top to bottom. Read each discussion prompt, review the example, and then write your group's answer in the blank markdown cells.

## Setup
The CSV file should be in the same folder as this notebook.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
DATA_FILE = 'restaurant_operations_sample.csv'

# Part 1 — Loading Data

### Discussion prompts
1. Why is `pd.read_csv()` frequently used?
2. Why should analysts examine a dataset immediately after loading it?
3. What does `.info()` tell us?
4. Why are data types important?

In [ ]:
df = pd.read_csv(DATA_FILE)
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

In [ ]:
df.info()

### Simple interpretation
- `read_csv()` converts a common text-based file into a DataFrame.
- Immediate inspection helps detect wrong columns, missing values, and incorrect data types.
- `.info()` shows column names, non-null counts, and data types.
- A wrong data type can cause calculations, comparisons, or date operations to fail.

### Group response
_Write your group's answer here._

# Part 2 — Exploring a DataFrame

Compare `.head()`, `.tail()`, `.sample()`, `.info()`, and `.describe()`.

In [ ]:
display(df.head(3))
display(df.tail(3))
display(df.sample(3, random_state=42))

In [ ]:
df.describe(include='all').T

### Discussion guide
- `.head()` checks the first records and column layout.
- `.tail()` checks the last records and can reveal file-ending problems.
- `.sample()` gives a less biased look at records from different parts of the file.
- `.describe()` summarizes numeric distributions and, with `include='all'`, categorical columns.

**Which method would your team use first, and why?**

_Write your answer here._

# Data Cleaning Before Further Analysis

The sample dataset intentionally contains missing values and inconsistent text.

In [ ]:
# Count missing values by column
df.isna().sum().sort_values(ascending=False)

In [ ]:
# Inspect category spelling/capitalization
df['weather'].value_counts(dropna=False)

In [ ]:
# Work on a copy so the original loaded data remains available
clean = df.copy()

# Standardize text values
clean['weather'] = clean['weather'].str.strip().str.title()

# Convert numeric-looking columns safely
numeric_cols = [
    'sales', 'transactions', 'labor_hours', 'customer_traffic',
    'avg_drive_through_min', 'customer_satisfaction',
    'rain_inches', 'temperature_f'
]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')

# Fill missing values with a simple median-based method
clean['labor_hours'] = clean['labor_hours'].fillna(clean['labor_hours'].median())
clean['customer_satisfaction'] = clean['customer_satisfaction'].fillna(
    clean['customer_satisfaction'].median()
)

clean.isna().sum()

# Part 3 — Working with Dates

Dates should not remain plain text because string values do not support reliable sorting, date arithmetic, or `.dt` components.

In [ ]:
clean['business_date'] = pd.to_datetime(clean['business_date'])
clean['timestamp'] = pd.to_datetime(clean['timestamp'])

clean['month'] = clean['business_date'].dt.month_name()
clean['weekday'] = clean['business_date'].dt.day_name()
clean['is_weekend'] = clean['business_date'].dt.dayofweek >= 5
clean['analysis_hour'] = clean['timestamp'].dt.hour

clean[['business_date', 'timestamp', 'month', 'weekday', 'is_weekend', 'analysis_hour']].head()

In [ ]:
# Examples of business analysis by weekday and hour
sales_by_weekday = clean.groupby('weekday', sort=False)['sales'].sum().sort_values(ascending=False)
sales_by_hour = clean.groupby('analysis_hour')['sales'].mean()

display(sales_by_weekday)
display(sales_by_hour)

### Group discussion
Give two business questions that could be answered by month, weekday, or hour.

Examples:
- Which weekday has the highest total sales?
- During which hour is the average drive-through time longest?

_Write two additional examples here._

# Part 4 — Selecting and Filtering Data

- Use `.iloc[]` for integer positions.
- Use `.loc[]` for labels and Boolean conditions.
- Conditional filtering avoids manually searching through many rows.

In [ ]:
# iloc: first 5 rows and first 4 columns
clean.iloc[:5, :4]

In [ ]:
# loc: selected columns for rows 0 through 4
clean.loc[0:4, ['business_date', 'sales', 'weather']]

In [ ]:
# Breakfast hours only: 7:00 through 10:59
breakfast = clean.loc[clean['analysis_hour'].between(7, 10)]

# Transactions greater than 100
high_transactions = clean.loc[clean['transactions'] > 100]

# Rainy days only
rainy_days = clean.loc[clean['weather'].eq('Rainy')]

# Weekend records
weekends = clean.loc[clean['is_weekend']]

print('Breakfast rows:', len(breakfast))
print('Transactions > 100 rows:', len(high_transactions))
print('Rainy rows:', len(rainy_days))
print('Weekend rows:', len(weekends))
display(high_transactions.head())

### Practice
Create a filter for **rainy weekend records with sales greater than $500**.

In [ ]:
# Write your solution below
rainy_weekend_high_sales = clean.loc[
    clean['weather'].eq('Rainy')
    & clean['is_weekend']
    & clean['sales'].gt(500)
]
rainy_weekend_high_sales

# Part 5 — Updating Data

Creating new columns usually preserves the original values and makes the calculation transparent.

In [ ]:
# Create calculated columns
clean['sales_per_labor_hour'] = clean['sales'] / clean['labor_hours']
clean['average_ticket'] = clean['sales'] / clean['transactions']

# Create a simple service flag
clean['service_status'] = 'On Target'
clean.loc[clean['avg_drive_through_min'] > 5.0, 'service_status'] = 'Needs Review'

clean[['sales', 'labor_hours', 'sales_per_labor_hour', 'average_ticket',
       'avg_drive_through_min', 'service_status']].head()

### Discussion guide
Possible calculated columns include:
- Sales per labor hour
- Average ticket amount
- Transactions per employee hour
- Weekend flag
- Service-performance flag

Overwriting original values can remove evidence, make errors harder to trace, and prevent comparison with the source. Direct updates are appropriate when correcting a verified error, standardizing categories, or applying a documented business rule.

_Write one additional calculated column your group recommends._

# Part 6 — Team Analysis Workflow

Management requests daily sales, labor hours, average drive-through time, customer satisfaction, and weather conditions.

In [ ]:
daily_report = (
    clean.groupby('business_date', as_index=False)
         .agg(
             daily_sales=('sales', 'sum'),
             labor_hours=('labor_hours', 'sum'),
             avg_drive_through_min=('avg_drive_through_min', 'mean'),
             avg_customer_satisfaction=('customer_satisfaction', 'mean'),
             weather=('weather', lambda s: s.mode().iat[0] if not s.mode().empty else 'Unknown'),
             rain_inches=('rain_inches', 'max')
         )
)

daily_report['sales_per_labor_hour'] = (
    daily_report['daily_sales'] / daily_report['labor_hours']
)

daily_report.round(2)

In [ ]:
# Accuracy checks before presentation
checks = {
    'duplicate_rows': clean.duplicated().sum(),
    'missing_values_total': int(clean.isna().sum().sum()),
    'negative_sales': int((clean['sales'] < 0).sum()),
    'invalid_satisfaction': int((~clean['customer_satisfaction'].between(1, 5)).sum()),
    'invalid_drive_time': int((clean['avg_drive_through_min'] <= 0).sum()),
}
pd.Series(checks, name='count')

### Recommended order
1. Load the dataset.
2. Inspect shape, columns, samples, nulls, and data types.
3. Clean text, numeric fields, missing values, and dates.
4. Create useful date and calculated columns.
5. Filter and validate records.
6. Aggregate the requested management metrics.
7. Perform reasonableness and accuracy checks.
8. Present the results with clear definitions.

### Essential Pandas functions
`read_csv`, `head`, `sample`, `info`, `describe`, `isna`, `to_numeric`, `to_datetime`, `loc`, `iloc`, `groupby`, `agg`, and column assignment.

### Group response
What additional validation would your team perform before presenting the report?

_Write your answer here._

# Reflection
Choose the most valuable beginner skill and explain why:
- Loading data
- Exploring data
- Selecting data
- Handling dates
- Updating data

_Write 3–5 sentences here._

# Optional Challenge
Find the single record with the highest sales and report its date, hour, weather, transactions, and labor hours.

In [ ]:
clean.loc[
    clean['sales'].idxmax(),
    ['business_date', 'analysis_hour', 'weather', 'sales', 'transactions', 'labor_hours']
]